# Phase 4 — Explore Custom Configurations

Test advanced techniques that go beyond LangChain's defaults:
- **LLM-based chunking** — AI generates headlines, summaries, and preserves original text
- **Reranking** — LLM reorders retrieval results by relevance
- **Query rewriting** — LLM reformulates the user's question for better retrieval

> **Run from the project root:** `uv run jupyter lab`

In [16]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

load_dotenv(override=True)

MODEL = "gpt-4.1-mini"
DB_NAME = "./vector_db_exploration_phase_4"
collection_name = "docs"
# embedding_model = "text-embedding-3-small"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("../knowledge-base")
AVERAGE_CHUNK_SIZE = 1000

openai = OpenAI()

## Data models

In [2]:
class Result(BaseModel):
    page_content: str
    metadata: dict


class Chunk(BaseModel):
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query"
    )
    summary: str = Field(
        description="A few sentences summarizing the content of this chunk to answer common questions"
    )
    original_text: str = Field(
        description="The original text of this chunk from the provided document, exactly as is, not changed in any way"
    )

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(
            page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,
            metadata=metadata,
        )


class Chunks(BaseModel):
    chunks: list[Chunk]

## Load documents

In [7]:
def fetch_documents():
    documents = []
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})
    print(f"Loaded {len(documents)} documents")
    return documents

all_documents = fetch_documents()

Loaded 76 documents


## LLM-based chunking

Instead of splitting by character count, we ask the LLM to chunk intelligently.
Each chunk gets a headline, summary, and the verbatim original text.

In [8]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

def make_messages(document):
    return [{"role": "user", "content": make_prompt(document)}]

# Preview the prompt for the first document
print(make_prompt(all_documents[0])[:500] + "...")


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: ../knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document shoul...


In [9]:
def process_document(document):
    messages = make_messages(document)
    response = openai.beta.chat.completions.parse(model=MODEL, messages=messages, response_format=Chunks)
    doc_as_chunks = response.choices[0].message.parsed.chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

# Test on first document
sample_chunks = process_document(all_documents[0])
print(f"Document split into {len(sample_chunks)} chunks")
print(f"\nFirst chunk preview:\n{sample_chunks[0].page_content[:300]}...")

Document split into 3 chunks

First chunk preview:
Founding and Initial Growth of Insurellm

Insurellm was founded in 2015 by Avery Lancaster as an insurance tech startup aimed at innovating the insurance sector. The company began with its first product, Markellm, a marketplace connecting consumers to insurance providers. It experienced rapid growth...


In [10]:
# Process all documents (this makes many LLM calls — takes a few minutes)
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

all_chunks = create_chunks(all_documents)
print(f"Total chunks created: {len(all_chunks)}")

100%|██████████| 76/76 [26:14<00:00, 20.72s/it]

Total chunks created: 350


## Create embeddings for LLM-enhanced chunks

In [17]:

def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]
    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

# Fall back to sample_chunks if cell-9 (process all documents) was skipped
chunks_to_embed = all_chunks if "all_chunks" in vars() else sample_chunks
if "all_chunks" not in vars():
    print("Note: using single-document sample (skip cell-9 warning expected)")
create_embeddings(chunks_to_embed)

Vectorstore created with 350 documents


## Visualize the LLM-enhanced embedding space

In [18]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
doc_texts = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [
    ['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)]
    for t in doc_types
]

In [19]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, doc_texts)],
    hoverinfo='text'
)])
fig.update_layout(
    title='2D — LLM-Enhanced Chunks',
    width=800, height=600, margin=dict(r=20, b=10, l=10, t=40)
)
fig.show()

In [20]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, doc_texts)],
    hoverinfo='text'
)])
fig.update_layout(
    title='3D — LLM-Enhanced Chunks',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900, height=700, margin=dict(r=10, b=10, l=10, t=40)
)
fig.show()

## Experiment: Reranking

After vector retrieval, ask an LLM to re-order results by true relevance.

In [ ]:
RETRIEVAL_K = 10

class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )


def rerank(question, chunks):
    system_prompt = """You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked."""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = openai.beta.chat.completions.parse(model=MODEL, messages=messages, response_format=RankOrder)
    order = response.choices[0].message.parsed.order
    return [chunks[i - 1] for i in order]


def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [ ]:
# Compare ranked vs unranked for a test question
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

print("=== Before reranking ===")
for i, chunk in enumerate(chunks):
    print(f"{i+1}. {chunk.page_content[:80]}...")

reranked = rerank(question, chunks)

print("\n=== After reranking ===")
for i, chunk in enumerate(reranked):
    print(f"{i+1}. {chunk.page_content[:80]}...")

## Experiment: Query rewriting

Reformulate the user's conversational question into a precise KB search query.

In [ ]:
def rewrite_query(question, history=[]):
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = openai.chat.completions.create(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content


# Test query rewriting
original = "Who won the IIOTY award?"
rewritten = rewrite_query(original)
print(f"Original:  {original}")
print(f"Rewritten: {rewritten}")

## Full advanced pipeline demo

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""


def make_rag_messages(question, history, chunks):
    context = "\n\n".join(
        f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks
    )
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]


def answer_question(question, history=[]):
    query = rewrite_query(question, history)
    print(f"Rewritten query: {query}")
    chunks = fetch_context_unranked(query)
    reranked = rerank(question, chunks)
    messages = make_rag_messages(question, history, reranked)
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content, reranked


answer, ctx = answer_question("Who won the IIOTY award?")
print(f"\nAnswer: {answer}")